# BinaryMatchboxNet KWS — Colab bootstrap

Run the cells top to bottom. Cell 1 always hard-resets the working copy to
`origin/main`, so **anything edited inside Colab is discarded** — edit locally,
commit, push, then re-run cell 1.

Prereq: a GitHub PAT stored as a Colab secret named `GH_TOKEN`
(🔑 icon in the left sidebar → *Add new secret* → toggle *Notebook access*).
See `docs/colab_setup.md`.

In [ ]:
# --- 1. clone / sync the repo -------------------------------------------
import os
from google.colab import userdata

GH_TOKEN = userdata.get('GH_TOKEN')
USER   = 'minochichic'
REPO   = 'KWS-AFE-Digital'
BRANCH = 'main'
DIR    = f'/content/{REPO}'

if not os.path.exists(DIR):
    !git clone -b {BRANCH} https://{GH_TOKEN}@github.com/{USER}/{REPO}.git {DIR}

%cd {DIR}
!git fetch origin
!git reset --hard origin/{BRANCH}
!git log -1 --oneline

In [ ]:
# --- 2. dependencies -----------------------------------------------------
# requirements-colab.txt deliberately omits torch/torchaudio: Colab's builds
# are matched to its CUDA runtime and reinstalling them breaks GPU support.
!pip install -q -r requirements-colab.txt

In [ ]:
# --- 3. environment check ------------------------------------------------
import torch, torchaudio, sys
print('python     ', sys.version.split()[0])
print('torch      ', torch.__version__)
print('torchaudio ', torchaudio.__version__)
print('cuda       ', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU only)')

if not torch.cuda.is_available():
    print('\n[!] No GPU. Runtime -> Change runtime type -> T4/A100 GPU.')

In [ ]:
# --- 4. unit tests (binary ops, config guardrails) -----------------------
!python -m pytest -q

In [ ]:
# --- 5. what will actually be built --------------------------------------
!python experiments/inspect_config.py configs/base.yaml

In [ ]:
# --- 6. persist checkpoints to Drive (optional) --------------------------
# Colab sessions are wiped on disconnect. Symlink runs/ into Drive so a long
# sweep survives a dropped runtime. Checkpoints are gitignored on purpose --
# do not push them.
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/kws_runs
!rm -rf runs && ln -s /content/drive/MyDrive/kws_runs runs
!ls -la runs/

---
## Not yet implemented (FIRST_TASK.md step 6)

The cells below are placeholders. Steps 2–5 (binary ops, AFE, model assembly,
overfit check) come first; only then the real dataset.

In [ ]:
# --- 7. dataset (step 6 — not implemented yet) ---------------------------
# Speech Commands v2 is ~2.3 GB. Download it to /content (fast local disk),
# NOT to Drive (slow) and NOT into the repo (gitignored anyway).
# !python -m data.download --root /content/datasets/speech_commands_v2

In [ ]:
# --- 8. training (step 5+ — not implemented yet) -------------------------
# !python -m train.train --config configs/base.yaml
# !python -m experiments.sweep --config configs/base.yaml --C 16 32 48 64 --T 40 64 96 128